# Bab 28: Etika, Privasi, dan Tata Kelola Data

Notebook pendamping buku *Data Mining dengan Python: Konsep, Algoritme, dan Studi Kasus dengan CRISP-DM*. Setiap sel kode sama dengan listing di buku; keluaran yang dicetak di buku ada di berkas `.out` di folder `kode/bab28/`.

Jalankan sel secara berurutan.

In [ ]:
# Jalankan sel ini sekali di awal sesi.
import os, sys, subprocess
if "google.colab" in sys.modules and not os.path.exists("kode"):
    if not os.path.exists("data-mining-dengan-python"):
        subprocess.run(["git", "clone", "--depth", "1",
                        "https://github.com/bsnegara/data-mining-dengan-python"])
    os.chdir("data-mining-dengan-python")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "-r", "kode/requirements-colab.txt"])
while not os.path.exists("kode/data") and os.getcwd() != "/":
    os.chdir("..")                     # cari akar repositori
sys.path.insert(0, "kode/bab28")
print("Folder kerja:", os.getcwd())

## Listing 28.1: k-anonimitas, pseudonim, dan privasi diferensial

`kode/bab28/anonimisasi.py`

In [ ]:
import hashlib
import hmac
import numpy as np
import pandas as pd

d = pd.read_csv("kode/data/pelanggan.csv")
kuasi = ["umur", "jenis_kelamin", "gerai_utama",
         "tanggal_daftar"]


def ukuran_kelas(df, kolom):
    grup = df.groupby(kolom, observed=True)
    return grup[kolom[0]].transform("size")


k = ukuran_kelas(d, kuasi)
print(f"anggota yang unik (k = 1): {(k == 1).mean():.1%}")

g = d.copy()
g["umur"] = pd.cut(g.umur, [16, 24, 34, 44, 54, 99],
                   labels=["17-24", "25-34", "35-44", "45-54", "55+"])
g["tanggal_daftar"] = g.tanggal_daftar.str[:4]          # tahun saja
g["gerai_utama"] = g.kota                               # kota saja
for nama, kolom in [("umur, jk, kota, tahun", kuasi),
                    ("umur, jk, kota", kuasi[:3])]:
    k = ukuran_kelas(g, kolom)
    print(f"generalisasi ({nama}): k minimum {k.min()}, "
          f"k < 5: {(k < 5).mean():.1%}")
aman = g[k >= 5]
print(f"setelah {len(g) - len(aman)} baris dengan k < 5 dihapus: "
      f"k minimum {ukuran_kelas(aman, kuasi[:3]).min()}")

KUNCI = b"simpan-di-brankas-rahasia"


def pseudonim(x):
    kode = hmac.new(KUNCI, x.encode(), hashlib.sha256)
    return kode.hexdigest()[:12]


print("\npseudonim:", d.id_anggota[:3].map(pseudonim).tolist())
polos = hashlib.sha256(b"A00001").hexdigest()[:12]
tebak = {hashlib.sha256(f"A{i:05d}".encode()).hexdigest()[:12]:
         f"A{i:05d}" for i in range(1, 100000)}
print(f"hash tanpa kunci {polos} dibalik: {tebak[polos]}")

rng = np.random.default_rng(28)
benar = int((d.kota == "Batam").sum())
for eps in (0.1, 1.0):
    jawab = benar + rng.laplace(0, 1 / eps, 5)
    print(f"jumlah anggota Batam (epsilon {eps}): "
          f"{np.round(jawab).astype(int).tolist()} (benar {benar})")

## Listing 28.2: Audit keadilan model churn

`kode/bab28/keadilan.py`

In [ ]:
import sys; sys.modules.pop('siapkan', None); sys.path.insert(0, 'kode/bab28')
import sys
import joblib
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score
sys.path.insert(0, "kode/bab27")
from siapkan import DATA, buat_model, fitur  # noqa: E402

model = joblib.load("kode/bab27/model/churn_v1.joblib")
d = pd.read_csv(DATA + "pelanggan_2026q3.csv")
d = d[d.kelompok == "kontrol"].reset_index(drop=True)
d["p"] = model.predict_proba(fitur(d, "2026-09-30"))[:, 1]
d["sasar"] = d.p >= 0.278


def ringkas(df, grup):
    baris = {}
    for g, x in df.groupby(grup):
        y, s = x.churn == 1, x.sasar
        baris[g] = {"churn": y.mean(), "skor": x.p.mean(),
                    "disasar": s.mean(), "TPR": s[y].mean(),
                    "FPR": s[~y].mean(), "presisi": y[s].mean()}
    hasil = pd.DataFrame(baris).T
    hasil.insert(0, "n", df.groupby(grup).size())
    return hasil


pd.set_option("display.float_format", "{:.3f}".format)  # noqa
print(ringkas(d, "jenis_kelamin").to_string())
print()
print(ringkas(d, "kota").sort_values("churn").to_string())

lama = pd.read_csv(DATA + "pelanggan.csv")
kat = ["kota", "tingkat"]
tanpa = buat_model(kategorik=kat).fit(
    fitur(lama, "2026-03-31", kat), lama.churn)
p2 = tanpa.predict_proba(fitur(d, "2026-09-30", kat))[:, 1]
auc = [roc_auc_score(d.churn, q) for q in (d.p, p2)]
print(f"\nAUC dengan jenis kelamin {auc[0]:.4f}, tanpa {auc[1]:.4f}")
print(f"korelasi skor: {np.corrcoef(d.p, p2)[0, 1]:.4f}; keputusan "
      f"berbeda: {((p2 >= 0.278) != d.sasar).sum()} anggota")

## Listing 28.3: Penjelasan global dan lokal

`kode/bab28/interpretasi.py`

In [ ]:
import sys; sys.modules.pop('siapkan', None); sys.path.insert(0, 'kode/bab28')
import sys
import warnings
import joblib
import lightgbm as lgb
import numpy as np
import pandas as pd
import shap
from sklearn.inspection import permutation_importance
from sklearn.metrics import roc_auc_score
sys.path.insert(0, "kode/bab27")
from siapkan import DATA, NUMERIK, fitur  # noqa: E402
warnings.simplefilter("ignore")

lama = pd.read_csv(DATA + "pelanggan.csv")
X = fitur(lama, "2026-03-31")
d = pd.read_csv(DATA + "pelanggan_2026q3.csv")
d = d[d.kelompok == "kontrol"].reset_index(drop=True)
Xu = fitur(d, "2026-09-30")
Xl_num, Xu_num = X[NUMERIK], Xu[NUMERIK]
gbm = lgb.LGBMClassifier(n_estimators=800, learning_rate=0.02,
                         num_leaves=4, reg_lambda=10, verbose=-1,
                         random_state=0).fit(Xl_num, lama.churn)
p_gbm = gbm.predict_proba(Xu_num)[:, 1]
print(f"LightGBM (atribut numerik) AUC uji: "
      f"{roc_auc_score(d.churn, p_gbm):.3f}")

nilai = shap.TreeExplainer(gbm).shap_values(Xu_num)
nilai = nilai[1] if isinstance(nilai, list) else nilai
glob = pd.Series(np.abs(nilai).mean(0), NUMERIK).sort_values()[::-1]
print("\nrata-rata |SHAP| (log-odds), enam terbesar:")
print(glob[:6].round(3).to_string())
g8 = glob[:8][::-1].rename("shap")
g8.index = g8.index.str.replace("_", r"\_")
g8.to_csv("gambar/bab28/shap_global.dat", sep=" ",
          index_label="atribut")

model = joblib.load("kode/bab27/model/churn_v1.joblib")
pi = permutation_importance(model, Xu, d.churn, scoring="roc_auc",
                            n_repeats=10, random_state=0)
pim = pd.Series(pi.importances_mean, Xu.columns).sort_values()[::-1]
print("\npermutation importance logistik (turunnya AUC):")
print(pim[:5].round(4).to_string())

i = int(np.argmax(p_gbm * (d.frekuensi_90h > 5)))
dasar = shap.TreeExplainer(gbm).expected_value
dasar = float(np.ravel(dasar)[-1])
s = pd.Series(nilai[i], NUMERIK)
top = s.reindex(s.abs().sort_values()[::-1].index)[:5]
print(f"\nanggota {d.id_anggota[i]}: log-odds dasar {dasar:.2f}, "
      f"akhir {dasar + s.sum():.2f} "
      f"(p = {1 / (1 + np.exp(-(dasar + s.sum()))):.2f})")
for k, v in top.items():
    print(f"  {k:18s} = {Xu_num.loc[i, k]:>9.2f}  SHAP {v:+.2f}")